JSONL: JSON Lines <br>
Her satırda bir adet JSON obje bulunur. <br>
Örneğin dil modeli eğitimi için şöyle görünür: <br>
{"prompt": "Soru: Teknolojinin hayatımıza kattığı kolaylıklar...", "response": "Cevap: A"} <br>
{"prompt": "Soru: Bir toplumun sanat anlayışı...", "response": "Cevap: A"} <br>
Bu format özellikle T5, LLaMA, GPT-neoX, Falcon, Mistral, Gemma, LoRA fine-tuning gibi modeller için idealdir.
### ------------------------------------------------------------------------------------------------------------
Eğer büyük bir veriyi klasik .json ile açarsan: tüm veriyi belleğe yüklemeye çalışır, RAM taşar. Ama .jsonl dosyası ile okunursa satır satır, yani akış halinde (streaming) okunur. Bu, özellikle model eğitimi(fine-tuning) sırasında GPU'ya veri akışı için mükemmeldir.

In [13]:
import json
import re
from pathlib import Path

input_path  = Path("../data/processed/turkce_questions.json")
output_path = Path("../data/processed/turkce_questions.jsonl")

OPTIONS_ORDER = ["A", "B", "C", "D", "E"]
PLACEHOLDER_ANSWER = "?"

def one_line(s: str) -> str:
    if s is None:
        return ""
    s = s.replace("\r", " ").replace("\n", " ")
    s = " ".join(s.split()).strip()

    # hece bölünmesini birleştir: "biçem- le" -> "biçemle"
    s = re.sub(r"(\w)-\s+(\w)", r"\1\2", s)

    # tek harf kopmalarını düzeltmeye çalış: "aylaV rında" -> "aylarında"
    # (çok agresif yapmıyoruz; sadece büyük harf + boşluk + küçük harf gibi kopmaları birleştiriyoruz)
    s = re.sub(r"([a-zçğıöşü])([A-ZÇĞİÖŞÜ])\s+([a-zçğıöşü])", r"\1\2\3", s)

    return s
def clean_leaks(s: str) -> str:
    """
    Seçenekler bittikten sonra prompt'a sızan tek harf/sayı artefaktlarını temizler.
    Örn: "... E) ... 1 Doğru seçenek ..." -> "... E) ... Doğru seçenek ..."
         "... E) ... A Doğru seçenek ..." -> "... E) ... Doğru seçenek ..."
    """
    s = re.sub(r"\s+[A-E]\s+(Doğru seçenek hangisidir\?)", r" \1", s)
    s = re.sub(r"\s+\d{1,2}\s+(Doğru seçenek hangisidir\?)", r" \1", s)
    return s

def build_options_text(secenekler: dict) -> str:
    lines = []
    for k in OPTIONS_ORDER:
        v = secenekler.get(k)
        if v:
            lines.append(f"{k}) {one_line(v)}")
    return "\n".join(lines).strip()

with open(input_path, "r", encoding="utf-8") as f:
    data = json.load(f)

written = 0
skipped_year = 0

with open(output_path, "w", encoding="utf-8") as f:
    for item in data:
        yil = item.get("yil")
        if not yil:
            skipped_year += 1
            continue

        soru_no = item.get("soru_no")
        soru = one_line(item.get("soru", ""))
        secenekler = item.get("secenekler", {}) or {}
        options_text = build_options_text(secenekler)

        prompt = (
            f"[Yıl: {yil} | Soru No: {soru_no}]\n"
            f"Soru: {soru}\n"
            f"Seçenekler:\n{options_text}\n"
            "Doğru seçenek hangisidir? (Sadece A/B/C/D/E yaz)"
        ).strip()

        prompt = clean_leaks(prompt)

        response = (item.get("cevap") or "").strip()
        if not response:
            response = PLACEHOLDER_ANSWER

        out = {
            "prompt": prompt,
            "response": response,
            "id": item.get("id"),
            "yil": yil,
            "soru_no": soru_no
        }

        f.write(json.dumps(out, ensure_ascii=False) + "\n")
        written += 1

print("✅ JSONL oluşturuldu:", output_path)
print("Yazılan:", written, "| Yıl boş atlanan:", skipped_year)


✅ JSONL oluşturuldu: ..\data\processed\turkce_questions.jsonl
Yazılan: 255 | Yıl boş atlanan: 2


In [14]:
import json

with open("../data/processed/turkce_questions.jsonl", "r", encoding="utf-8") as f:
    first = f.readline()

json.loads(first)   # hata vermezse JSONL formatın doğru


{'prompt': '[Yıl: 2006 | Soru No: 3]\nSoru: Su altı zenginlikleriyle öne çıkan Kuzey Kıbrıs, dalış sporuna ilgi duyanlar için birçok seçenek sunuyor. I Kıbrıs’ı barınak seçmiş iki binin üzerinde Caretta II caretta kaplumbağasını ya da beş yüz kadar yeşil III kaplumbağayı, bu kıyılarda hemen hemen bütün bir IV yıl boyunca yapılabilen dalışlarda görmek mümkün. Deniz kaplumbağaları Gazi Magosa’yı yıl boyunca hiç terk etmiyor. Öteki plajlardaysa yalnızca yaz aylaVrında görülebiliyor. Bu parçadaki numaralanmış sözlerin hangisinde kesinlik söz konusudur?\nSeçenekler:\nA) I.\nB) II.\nC) III.\nD) IV.\nE) V.\nDoğru seçenek hangisidir? (Sadece A/B/C/D/E yaz)',
 'response': '?',
 'id': '2006_3',
 'yil': 2006,
 'soru_no': 3}

In [15]:
import json

with open("../data/processed/turkce_questions.jsonl", "r", encoding="utf-8") as f:
    obj = json.loads(f.readline())

print(obj["prompt"])  # ekranda alt alta görmen lazım


[Yıl: 2006 | Soru No: 3]
Soru: Su altı zenginlikleriyle öne çıkan Kuzey Kıbrıs, dalış sporuna ilgi duyanlar için birçok seçenek sunuyor. I Kıbrıs’ı barınak seçmiş iki binin üzerinde Caretta II caretta kaplumbağasını ya da beş yüz kadar yeşil III kaplumbağayı, bu kıyılarda hemen hemen bütün bir IV yıl boyunca yapılabilen dalışlarda görmek mümkün. Deniz kaplumbağaları Gazi Magosa’yı yıl boyunca hiç terk etmiyor. Öteki plajlardaysa yalnızca yaz aylaVrında görülebiliyor. Bu parçadaki numaralanmış sözlerin hangisinde kesinlik söz konusudur?
Seçenekler:
A) I.
B) II.
C) III.
D) IV.
E) V.
Doğru seçenek hangisidir? (Sadece A/B/C/D/E yaz)
